# Module 1 - Few-Shot Prompting

---

## What you will be able to craft from this module:

- A real understanding of what few-shot prompting is and why it works differently from zero-shot
- A mixed dataset built from real-world phishing data and AI-generated examples
- Your own detection prompt — written from scratch, tested, and iterated
- A side-by-side comparison of zero-shot vs few-shot vs your own prompt
- A real production scenario grounded in an actual threat that companies face today

---

## Before you start

Make sure you have the following requirements ready:
- Completed Module 0, some concepts here build directly on what we did there
- Python 3.10+
- A free Groq API key from [console.groq.com](https://console.groq.com)
- A `.env` file in the root of the repo with `GROQ_API_KEY=gsk_...`
- Dependencies installed: `pip install -r requirements.txt` from the root file

---

# The plan

You just started as the first dedicated security engineer at **Owlsight**, a cybersecurity startup with let's say... 120 people that sells threat monitoring and anomaly detection.

So in the basics you sell security to others. Now you have to build it for yourself.

You have just started your week, but three employees have forwarded you suspicious emails. One of them already clicked a link.

Your job today: build something that catches these before they reach the inbox.

---

# The Threat | AI-Generated Phishing

I really think about which focus to give on this module, not only as a follow up from the previous one but to actually be interesting, and phishing has been around for ages now, but AI phishing, kinda rings a bell right? let's continue this pace of lets built AI with AI againts AI for the sake of AI, at the end if we really want to learn about something you need to deep dive into that.

## So why traditional filters are now failing

Phishing detection relied on signals that are now completely dead:

- **Bad grammar and spelling:** LLMs write better English than most humans
- **Generic lures:** "Dear customer" is gone, replaced by emails that reference your actual job title, your manager's name, your recent projects
- **Suspicious formatting:** AI-generated emails match the exact tone and structure of legitimate company communications
- **Known malicious domains:** attackers now use compromised legitimate domains or generate convincing lookalikes

The old playbook doesn't work anymore. And the volume is only going up.

## What you are actually dealing with at Owlsight

The three emails forwarded to you this morning represent three different attack patterns:

**Email 1 | CEO impersonation (BEC)**
> *From: daniel.reyes@owlsight-corp.com*
> *"Hey, I need you to review this contract before our board call at 3pm. It's sensitive so please don't loop in anyone else yet. Link below."*

That domain is not owlsight.com. It's owlsight-**corp**.com. Registered yesterday.

**Email 2 | IT helpdesk impersonation**
> *From: security-ops@owlsight.com*
> *"We detected an unusual login to your account from San Jose, Costa Rica. If this wasn't you, verify your identity immediately to prevent account suspension."*

Perfectly written. Correct branding. The link goes to a credential harvesting page.

**Email 3 | Vendor impersonation**
> *From: no-reply@docusign-secure.net*
> *"Your NDA with Meridian Capital Partners requires your signature by EOD. Click to review."*

Owlsight is actually in conversations with Meridian Capital. The attacker did their homework.

## Why few-shot is the right tool for this problem

Here's the challenge with phishing detection: **the threat evolves faster than you can retrain a model.**

A new phishing campaign emerges. You see 3 examples. You need your classifier to catch the next 10,000 variations of that same campaign today, not after a two-week retraining cycle.

That's exactly what few-shot prompting solves. You show the model 2-3 examples of what the new attack looks like, and it immediately generalizes. No retraining. No new dataset. Just a prompt update.

This is why now security teams at companies use few-shot prompting in their threat detection pipelines, not as a permanent solution, but as a faster response layer while the fine-tuned model catches up.

---

# But wait... We Already Used Few-Shot in Module 0? +10 points if you noticed

Let's pause on something.

In Module 0 when we built the dataset generation prompt, we included this:

```python
[
  {"text": "I finally finished my thesis after 3 years!", "label": "happy"},
  {"text": "The meeting was rescheduled to Thursday.", "label": "neutral"},
  {"text": "I missed the last train home.", "label": "sad"}
]
```

You saw those examples? Well that **was** few-shot prompting. We were showing the model the format, the style, and the label structure before asking it to generate more.

We used it without naming it. Now we're going to use it with a more higher purpose, as a defense mechanism.

The difference between Module 0 and Module 1 is pure intention. In Module 0 few-shot was a tool to get better output format. In Module 1, few-shot is the actual technique we're deploying to solve a real security problem.

## So what exactly is few-shot prompting?

Few-shot prompting means providing a small number of input-output examples directly in your prompt before asking the model to perform the task. Instead of just describing what you want, you show it.

```
# Zero-shot:
"Is this email phishing or legitimate?"
Email: "Your account has been compromised. Click here."

# Few-shot:
"Is this email phishing or legitimate? Here are some examples:"

Email: "Please review Q3 report attached" → legitimate
Email: "Your password expires in 24h. Reset now." → phishing
Email: "Hey, can we move our 1:1 to Thursday?" → legitimate

Now classify:
Email: "Your account has been compromised. Click here."
```

The examples anchor the model's behavior. They define what "phishing" and "legitimate" mean in your specific context, not in the abstract, but in the language and patterns of your actual environment.

## The "shot" terminology again

- **Zero-shot** = no examples, just the task description (Module 0)
- **One-shot** = one example per class
- **Few-shot** = typically 2-5 examples per class
- **Many-shot** = tens to hundreds of examples in the prompt (usually expensive, but powerful)
- **Fine-tuning** = training the model weights directly on your data (Module 2 territory)

The sweet spot for most production use cases is few-shot, enough examples to meaningfully guide the model, and in that way you make sure to not be burning tokens on every request.

---

# Setting Up the Lab

> 💡 **Location of this code block?** Same setup as Module 0, `shared/utils/api_helpers.py`. If you've already run Module 0 in this environment, your key is already loaded.

In [ ]:
import os
import json
import time
import random
import warnings
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv

# Find the repo root dynamically
notebook_dir = Path(os.getcwd())
repo_root = notebook_dir
while not (repo_root / ".env").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

load_dotenv(repo_root / ".env")

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise EnvironmentError(
        "GROQ_API_KEY not found. "
        "Make sure your .env file exists at the repo root and contains GROQ_API_KEY=gsk_..."
    )

print(f"✅ Groq API key loaded: {api_key[:8]}...")

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "llama-3.1-8b-instant"

print(f"✅ Client ready — using model: {MODEL}")

---

# Let's find a real Database - Part 2

## Why we'll mix real data with generated data

In Module 0 we generated 100% of our dataset using Groq. That worked fine for sentiment analysis.

Phishing is different. Real phishing emails have specific characteristics that a language model generating examples might not fully capture:
- Actual malicious URLs and domain patterns
- Real lure templates used by known threat actors
- The subtle language patterns that human analysts have flagged all these years

At the same time, public phishing datasets have a blind spot: **they don't contain AI-generated phishing**, because most of them were collected before LLMs became widely used by attackers. That's where Groq will be useful, we generate the AI-phishing examples that the real datasets are missing.

So our strategy is:
- **Real data** → traditional phishing and legitimate emails from the Kaggle corpus
- **Generated data** → AI-crafted phishing emails in the Owlsight context

# Why I choose the Kaggle Phishing Email Dataset?

After checking for a while I ended up finding this [Kaggle dataset](https://www.kaggle.com/datasets/naserabdullahalam/phishing-email-dataset), a collection of labeled real phishing and legitimate emails with full body text. Unlike URL-only databases, this one gives us the actual email content, which is exactly what we need.

**Other great reasons to be using this one:**
- Contains full email body text, not just URLs or metadata
- Labeled across phishing and legitimate categories
- Large enough to give us a solid foundation
- Free to download with a Kaggle account

**What it doesn't cover:**
- AI-generated phishing (too recent for any public big dataset)
- Internal corporate lures specific to your environment
- Spear-phishing with personalized context

That gap is exactly what we'll fill with our Groq Owlsight examples.

## Let's put some good practices into play: Evaluate your data before anything

Before you use any public dataset in production, ask these questions:

1. **Who collected it and how?** Human-verified is better than automated. Community-verified is better than single-source.
2. **How old is it?** Phishing tactics from 2015 look very different from 2024-2026.
3. **What's the label quality?** Are labels binary (phishing/not) or more granular? Do they cover the classes you care about?
4. **What's the class distribution?** A dataset with 95% phishing and 5% legitimate will produce a biased model.
5. **Is it representative of your environment?** A dataset for consumer banking phishing won't generalize well to B2B SaaS impersonation.

The Kaggle dataset scores well on 1, 2, and 3. Let's try to address 4 and 5 by balancing the distribution and adding our Owlsight Groq examples.

---

## Getting the Kaggle Dataset

You have two options here.

**Option A:** the hands-on path (the one I recommend), you go to the source, understand what you're downloading, and drop it in manually.

**Option B:** the code does it automatically. Both end up in the same place.

If you want to actually understand what's in your training data as you should, Option A is worth the five minutes.

---

### Option A | Download it yourself (recommended)

1. Go to [kaggle.com/datasets/naserabdullahalam/phishing-email-dataset](https://www.kaggle.com/datasets/naserabdullahalam/phishing-email-dataset)
2. Create a free Kaggle account if you don't have one
3. Click **Download** and unzip the file
4. Place `phishing_email.csv` at: `modules/01_few_shot/data/`
5. Come back and run the cell below

While you're on the page, scroll through the dataset card and read the description. Check the column names, the class distribution, the source. This is the habit that you need to start doing, know your data before you touch it.

---

### Option B | Download programmatically

You'll need a Kaggle API key for this. Go to [kaggle.com/settings](https://kaggle.com/settings) → **API** → **Create New Token**. This downloads a `kaggle.json` file, place it at `~/.kaggle/kaggle.json`. Then run the cell below.

In [ ]:
import subprocess

# Find the root repo dynamically then build the path from there
notebook_dir = Path(os.getcwd())
repo_root = notebook_dir
while not (repo_root / ".env").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

data_dir = repo_root / "modules" / "01_few_shot" / "data"
data_dir.mkdir(parents=True, exist_ok=True)

figures_dir = repo_root / "modules" / "01_few_shot" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = data_dir / "phishing_email.csv"
print(f"Data directory: {data_dir.resolve()}")

if OUTPUT_PATH.exists():
    print(f"✅ Dataset already exists at {OUTPUT_PATH}")
    print("   Delete the file and re-run this cell if you want a fresh download.")
else:
    print("Downloading Kaggle phishing email dataset...")
    print("Make sure kaggle.json is at ~/.kaggle/kaggle.json\n")

    try:
        subprocess.run(["pip", "install", "kaggle", "-q"], check=True)

        result = subprocess.run(
            ["kaggle", "datasets", "download",
             "naserabdullahalam/phishing-email-dataset",
             "--path", str(data_dir), "--unzip"],
            capture_output=True, text=True
        )

        if result.returncode == 0:
            csv_files = list(data_dir.glob("*.csv"))
            matching = [f for f in csv_files if "phishing_email" in f.name]
            if matching and matching[0].name != "phishing_email.csv":
                matching[0].rename(OUTPUT_PATH)
            print(f"✅ Downloaded to {OUTPUT_PATH}")
        else:
            print(f"❌ Download failed: {result.stderr}")
            print("   Try Option A — download manually from kaggle.com")

    except Exception as e:
        print(f"❌ Error: {e}")
        print("   Try Option A — download manually from kaggle.com")

## Exploring the dataset

Before we use any dataset, we look at it. Always. This is one of those habits that separates people who build reliable systems from people who wonder why their model behaves strangely six months later. We want to understand the shape of what we're working with before writing a single line of model code.

Run the cell below and you'll see the basic structure of it, how many rows, what columns exist, whether there are any null values, and what the first few rows actually look like. Try to not skip this.

> 💡 **Location of this code block?** This is exploratory analysis, it lives in the notebook only. The cleaned output feeds into `scripts/01_generate_dataset.py`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(OUTPUT_PATH)

print("Dataset overview:")
print(f"  Shape:   {df.shape}")
print(f"  Columns: {df.columns.tolist()}")
print(f"  Nulls:   {df.isnull().sum().sum()}")
print()
print("First 3 rows:")
print(df.head(3).to_string())

## Understanding the columns

The dataset has two columns: `text_combined` and `label`. Simple. `text_combined` is a merged field that combines the subject line, body, sender, and date into one string, which is actually useful for us since phishing signals can appear anywhere in an email. `label` is binary: `1` means phishing, `0` means legitimate.

The cell below confirms this and shows you the label distribution, how many phishing vs legitimate emails are in the full dataset. You'll notice it's not balanced, which is worth keeping in mind for any training model.

In [ ]:
# phishing_email.csv has exactly two columns: text_combined and label
# text_combined merges subject + body + sender into one field
# label could be 2 options: 1 = phishing, 0 = legitimate

TEXT_COL  = "text_combined"
LABEL_COL = "label"

print(f"Using text column:  '{TEXT_COL}'")
print(f"Using label column: '{LABEL_COL}'")
print()
print("Label distribution:")
print(df[LABEL_COL].value_counts())

## Mapping the labels

The dataset uses `1` and `0` as labels, which works for a binary classifier but isn't useful when we have three classes. We're going to map those values to human-readable names that match our classification task: `traditional_phishing` and `legitimate`.

This matters because our third class, `ai_phishing` doesn't exist in the Kaggle dataset at all. We're generating that ourselves with Groq. Think about this section as the bridge between the raw dataset and the three-class format we're building in a couple of minutes.

In [ ]:
label_map = {1: "traditional_phishing", 0: "legitimate"}

df["label_clean"] = df[LABEL_COL].map(label_map)

print(f"Label mapping: {label_map}")
print()
print("Cleaned distribution:")
print(df["label_clean"].value_counts())

## Visualizing what we have

Two charts, label distribution and email length. The first one tells you if the dataset is balanced enough to train on. The second one tells you how long these emails typically are, which matters for tokenization later, if most emails are 200 characters but we're truncating at 50, we're throwing away signal.

What to look for: are the two classes roughly comparable in size? Are the length distributions similar between phishing and legitimate, or does one class tend to be significantly longer? Both of those things affect how a model learns.

In [ ]:
colors = {"legitimate": "#2ecc71", "traditional_phishing": "#e74c3c"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df["label_clean"].value_counts()
axes[0].bar(counts.index, counts.values,
            color=[colors.get(l, "#95a5a6") for l in counts.index],
            edgecolor="white", linewidth=1.5)
axes[0].set_title("Label Distribution")
axes[0].set_ylabel("Count")
for i, (label, count) in enumerate(counts.items()):
    axes[0].text(i, count + 50, str(count), ha="center", fontsize=11)
axes[0].grid(axis="y", alpha=0.3)

df["text_len"] = df[TEXT_COL].str.len()
for label in ["legitimate", "traditional_phishing"]:
    subset = df[df["label_clean"] == label]["text_len"]
    axes[1].hist(subset.clip(upper=5000), bins=50, alpha=0.6,
                 color=colors[label], label=label)
axes[1].set_title("Email Length Distribution")
axes[1].set_xlabel("Characters")
axes[1].set_ylabel("Count")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(figures_dir / "dataset_overview.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nDataset stats:")
print(f"  Total emails:     {len(df)}")
print(f"  Avg email length: {df['text_len'].mean():.0f} chars")
print(f"  Max email length: {df['text_len'].max()} chars")

## Reading actual emails

No visualization replaces reading the data yourself. Run this cell and actually read what comes out, not just skim it. Look at a legitimate email and ask yourself: does it feel like something a real person wrote? Look at a phishing email and ask: what signals give it away?

This is important because your model will learn from these. If the data has problems, garbled text, emails in other languages, metadata mixed in with content, you need to catch that now before it becomes a training data problem.

In [ ]:
print("Sample LEGITIMATE emails:")
print("-" * 60)
for _, row in df[df["label_clean"] == "legitimate"].sample(3, random_state=42).iterrows():
    print(row[TEXT_COL][:300])
    print()

print("Sample TRADITIONAL PHISHING emails:")
print("-" * 60)
for _, row in df[df["label_clean"] == "traditional_phishing"].sample(3, random_state=42).iterrows():
    print(row[TEXT_COL][:300])
    print()

## Cleaning and sampling

Two things happening here. First, we filter out garbage emails that are too short, non-English text, and anything that's mostly non-ASCII characters (which usually means garbled encoding or metadata noise). A bad training example can hurt the model in a pretty bad way.

Second, we sample 100 examples per class. We don't need all 80k emails, we need a clean, balanced subset. Equal class sizes mean the model doesn't learn shortcuts based on frequency. Run this and check the final count at the bottom.

In [ ]:
SAMPLES_PER_CLASS = 100

def clean_text(text: str) -> str:
    """Basic text cleaning — normalize whitespace, filter non-English, truncate."""
    if not isinstance(text, str):
        return ""
    text = " ".join(text.split())
    if len(text) < 20:
        return ""
    ascii_ratio = sum(1 for c in text if ord(c) < 128) / len(text)
    if ascii_ratio < 0.85:
        return ""
    return text[:1000]


real_data = []

for label in ["legitimate", "traditional_phishing"]:
    subset = df[df["label_clean"] == label].copy()
    subset = subset.dropna(subset=[TEXT_COL])
    subset = subset[subset[TEXT_COL].str.len() > 20]
    n = min(SAMPLES_PER_CLASS, len(subset))
    sampled = subset.sample(n, random_state=42)

    for _, row in sampled.iterrows():
        text = clean_text(row[TEXT_COL])
        if len(text) > 20:
            real_data.append({"text": text, "label": label, "source": "kaggle"})

print(f"Real data prepared: {len(real_data)} examples")
for label, count in Counter(ex["label"] for ex in real_data).items():
    print(f"  {label}: {count}")

---

# Let's put Groq into work, let's generate our AI Phishing dataset

Now we generate the third class that no public dataset has yet: AI-crafted spear-phishing targeting Owlsight specifically.

This is where the Owlsight scenario becomes real training data. Every generated email references the actual company context, the people, the tools, the current events, because that's exactly what a sophisticated attacker would research before launching a campaign.

## The Owlsight cast

- **Daniel Reyes:** CEO
- **Priya Nair:** Head of People & Culture
- **IT & Security Ops:** internal helpdesk
- **Vendors:** Google Workspace, Stripe, AWS, GitHub, DocuSign

> 💡 **Location of this code block?** This is the base of `scripts/01_generate_dataset.py`, specifically the AI phishing generation section.

In [ ]:
OWLSIGHT_CONTEXT = """
Company: Owlsight, a 120-person cybersecurity SaaS company
Product: Threat monitoring and anomaly detection for mid-market enterprises
Email domain: @owlsight.com
People:
  - Daniel Reyes (CEO, daniel.reyes@owlsight.com)
  - Priya Nair (Head of People & Culture, priya.nair@owlsight.com)
  - IT & Security Ops (it-ops@owlsight.com)
Tools used: Google Workspace, Slack, GitHub, AWS, Stripe, DocuSign
Current events: Recently closed Series A, onboarding 3 new enterprise clients,
                hiring aggressively across engineering and sales
"""

AI_PHISHING_SYSTEM = """You are generating realistic examples of AI-generated spear-phishing emails
for a cybersecurity training dataset. These represent the new wave of sophisticated phishing:
personalized, well-written, context-aware attacks that reference specific company details,
real people's names, current events, and legitimate business contexts.
They are designed to bypass traditional filters and fool even security-aware employees.
This data will be used to train a phishing detection model. Generate realistic examples only.
Respond ONLY with a valid JSON array, no explanation, no markdown, no preamble."""

AI_PHISHING_USER = """Generate {n} AI-generated spear-phishing email examples targeting Owlsight employees.

Company context:
{context}

These emails should be sophisticated and hard to spot:
- Impersonate Daniel Reyes (CEO), Priya Nair (HR), IT Security Ops, or trusted vendors
- Reference real Owlsight context (Series A, new clients, tools they use)
- Use subtle urgency without obvious red flags
- Include plausible but fake domains (owlsight-corp.com, owlsight.security-portal.com)
- Target credential harvesting, wire transfers, malware delivery, or data exfiltration
- Write in a natural, professional tone, no spelling errors, no generic language

Each item must have exactly two fields: "text" and "label".
Label must be: "ai_phishing"

Example format:
[
  {{"text": "Hi, this is Daniel. I'm in back-to-back calls with the EMEA team but need you to review the updated NDA before 5pm. I've shared it here: docs.owlsight-secure.net/nda-v3, please don't forward, it's under NDA itself. Thanks", "label": "ai_phishing"}}
]

Generate exactly {n} examples. Keep each email under 100 words. Return ONLY the JSON array:"""

print("✅ AI phishing generation prompts defined")

In [ ]:
# Test the prompt with 2 examples before the full run
# Always validate before scaling up

test_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": AI_PHISHING_SYSTEM},
        {"role": "user",   "content": AI_PHISHING_USER.format(
            n=2, context=OWLSIGHT_CONTEXT
        ) + "\n\nIMPORTANT: Keep each email under 100 words."}
    ],
    temperature=0.9,
    max_tokens=2048,
)

raw = test_response.choices[0].message.content.strip()

# Always print the raw output first so you can see exactly what the model returned
# Common issues that you might encounter: trailing commas, smart quotes, markdown fences, incomplete JSON
print("=== RAW MODEL OUTPUT ===")
print(raw)
print("========================\n")

import re
if "```" in raw:
    parts = raw.split("```")
    raw = parts[1] if len(parts) > 1 else parts[0]
    if raw.startswith("json"):
        raw = raw[4:]

match = re.search(r'\[.*\]', raw, re.DOTALL)
if match:
    raw = match.group(0)

try:
    test_examples = json.loads(raw.strip())
except json.JSONDecodeError:
    print("⚠️ Full parse failed — extracting individual objects...\n")
    objects = re.findall(r'\{[^{}]+\}', raw, re.DOTALL)
    test_examples = [json.loads(obj) for obj in objects if '"text"' in obj]

print("Sample AI phishing emails:\n")
for ex in test_examples:
    print(f"  {ex['text']}")
    print()

In [ ]:
AI_PHISHING_TARGET = 100
BATCH_SIZE = 20

ai_phishing_examples = []
batch_num = 0
failures = 0

print(f"Generating {AI_PHISHING_TARGET} AI phishing examples...\n")

while len(ai_phishing_examples) < AI_PHISHING_TARGET:
    remaining = AI_PHISHING_TARGET - len(ai_phishing_examples)
    this_batch = min(BATCH_SIZE, remaining)
    batch_num += 1

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": AI_PHISHING_SYSTEM},
                {"role": "user",   "content": AI_PHISHING_USER.format(
                    n=this_batch, context=OWLSIGHT_CONTEXT
                )}
            ],
            temperature=0.9,
            max_tokens=2048,
        )

        raw = response.choices[0].message.content.strip()
        if "```" in raw:
            raw = raw.split("```")[1]
            if raw.startswith("json"): raw = raw[4:]

        batch = json.loads(raw.strip())

        valid = [
            {"text": ex["text"].strip(), "label": "ai_phishing", "source": "generated"}
            for ex in batch
            if isinstance(ex, dict)
            and "text" in ex
            and len(ex["text"].strip()) > 10
        ]

        ai_phishing_examples.extend(valid)
        print(f"  Batch {batch_num}: {len(valid)} valid examples (total: {len(ai_phishing_examples)}/{AI_PHISHING_TARGET})")

        if len(ai_phishing_examples) < AI_PHISHING_TARGET:
            time.sleep(2)

    except json.JSONDecodeError:
        failures += 1
        print(f"  Parse error on batch {batch_num}, retrying...")
        time.sleep(3)
        if failures > 5:
            print("  Too many failures, saving what we have.")
            break

    except Exception as e:
        if "rate_limit" in str(e).lower() or "429" in str(e):
            print("  Rate limit hit — waiting 60s...")
            time.sleep(60)
        else:
            raise

ai_phishing_examples = ai_phishing_examples[:AI_PHISHING_TARGET]
print(f"\n✅ Generated {len(ai_phishing_examples)} AI phishing examples")

In [ ]:
# Combine and save
dataset = real_data + ai_phishing_examples
random.shuffle(dataset)

print(f"Full dataset: {len(dataset)} examples")
for label, count in Counter(ex["label"] for ex in dataset).items():
    bar = "█" * (count // 3)
    print(f"  {label:25s} {count:4d}  {bar}")

def save_jsonl(examples: list, path: Path):
    with open(path, "w") as f:
        for ex in examples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print(f"  Saved {len(examples):4d} examples → {path}")

save_jsonl(dataset, data_dir / "phishing_raw.jsonl")
print(f"\n✅ Dataset saved to {data_dir.resolve()}")

---

# The main Few-Shot Demonstration

This is the actual core of the module. We'll run two experiments on the same set of test emails:
1. **Zero-shot:** just tell the model what to do, no examples
2. **Few-shot:** give it 2 examples per class before asking

The emails are designed to stress the hardest case: **AI-generated phishing**. These are the ones that fool people.

In [ ]:
TEST_EMAILS = [
    # --- Legitimate ---
    {"text": "Hey, quick heads up, the 3pm client call has been moved to 4pm. Same link. — Priya",
     "label": "legitimate"},
    {"text": "Team, we're enabling mandatory MFA for all GitHub repos starting Monday. IT-Ops will send setup instructions to your work email.",
     "label": "legitimate"},
    {"text": "Hi everyone, excited to announce we've officially signed our third enterprise client this quarter. Details in the all-hands Friday. — Daniel",
     "label": "legitimate"},

    # --- Traditional Phishing ---
    {"text": "YOUR ACCOUNT HAS BEEN SUSPENDED. Click here immediately to verify your identity and restore access: secure-paypa1.com/verify",
     "label": "traditional_phishing"},
    {"text": "Congratulations! You have been selected to receive a $500 Amazon gift card. Claim your reward now before it expires.",
     "label": "traditional_phishing"},
    {"text": "Dear Customer, your Microsoft account password will expire in 24 hours. Update it now to avoid losing access to your files.",
     "label": "traditional_phishing"},

    # --- AI Phishing (the hard ones) ---
    {"text": "Hey, it's Daniel. I'm in back-to-back calls with the Meridian team but need you to review the updated NDA before 5pm. Shared it here: docs.owlsight-secure.net/nda — please don't forward, it's confidential. Thanks",
     "label": "ai_phishing"},
    {"text": "Hi, this is Priya from People & Culture. As part of our Series A onboarding updates, we need all employees to re-verify their benefits enrollment by EOD. Use the HR portal: owlsight-hr.people-portal.com",
     "label": "ai_phishing"},
    {"text": "Security alert: we detected a login to your Owlsight GitHub account from an unrecognized device in Amsterdam. If this wasn't you, revoke access immediately: github-owlsight.security-verify.com/revoke",
     "label": "ai_phishing"},
]

print(f"✅ {len(TEST_EMAILS)} test emails ready")
print("\nClass distribution:")
for label, count in Counter(ex["label"] for ex in TEST_EMAILS).items():
    print(f"  {label}: {count}")

In [ ]:
ZERO_SHOT_SYSTEM = """You are a phishing detection system for Owlsight, a cybersecurity company.
Classify emails into exactly one of three categories:
- legitimate: real internal business email
- traditional_phishing: generic phishing with obvious red flags
- ai_phishing: sophisticated, personalized phishing that is hard to spot

Respond with exactly one of these three labels and nothing else."""


def classify_zero_shot(email_text: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": ZERO_SHOT_SYSTEM},
            {"role": "user",   "content": f"Classify this email:\n\n{email_text}"}
        ],
        temperature=0,
        max_tokens=10,
    )
    pred = response.choices[0].message.content.strip().lower()
    for label in ["ai_phishing", "traditional_phishing", "legitimate"]:
        if label in pred:
            return label
    return "legitimate"


print("Running zero-shot classification...\n")
zero_shot_results = []

for ex in TEST_EMAILS:
    pred = classify_zero_shot(ex["text"])
    zero_shot_results.append(pred)
    time.sleep(0.5)

zero_shot_acc = sum(p == e["label"] for p, e in zip(zero_shot_results, TEST_EMAILS)) / len(TEST_EMAILS)

for i, (ex, pred) in enumerate(zip(TEST_EMAILS, zero_shot_results)):
    correct = "✅" if pred == ex["label"] else "❌"
    status  = "CORRECT" if pred == ex["label"] else "WRONG"
    print(f"{correct} #{i+1} [{status}]")
    print(f"   Email:     {ex['text']}")
    print(f"   True:      {ex['label']}")
    print(f"   Predicted: {pred}")
    print()

print(f"Zero-shot accuracy: {zero_shot_acc*100:.0f}%")

## What you're seeing

Look at the ❌ ones, those are the interesting part.

Traditional phishing getting caught makes sense. Those patterns have been around for years and the model has seen thousands of examples during pretraining. Obvious urgency, suspicious domains, generic lures, it knows what those look like.

The AI phishing ones are where it breaks down. The model has no reference point for what a Daniel Reyes impersonation looks like, or how Owlsight's internal comms are structured. It's working purely from the description we gave it, and that description isn't enough.

That's exactly the gap we're about to close.

In [ ]:
def get_few_shot_examples(dataset: list, n_per_class: int = 2) -> list:
    """Pick n examples per class from our dataset to use as few-shot anchors."""
    examples = []
    for label in ["legitimate", "traditional_phishing", "ai_phishing"]:
        class_examples = [ex for ex in dataset if ex["label"] == label]
        examples.extend(random.sample(class_examples, min(n_per_class, len(class_examples))))
    return examples


few_shot_examples = get_few_shot_examples(dataset, n_per_class=2)

few_shot_block = "\n".join([
    f'Email: "{ex["text"][:200]}"\nLabel: {ex["label"]}\n'
    for ex in few_shot_examples
])

FEW_SHOT_SYSTEM = f"""You are a phishing detection system for Owlsight, a cybersecurity company.
Classify emails into exactly one of three categories:
- legitimate: real internal business email
- traditional_phishing: generic phishing with obvious red flags
- ai_phishing: sophisticated, personalized phishing that is hard to spot

Here are examples of each class:

{few_shot_block}
Respond with exactly one of the three labels and nothing else."""

print("Few-shot examples being used:\n")
for ex in few_shot_examples:
    print(f"  [{ex['label']}]")
    print(f"  {ex['text']}")
    print()

In [ ]:
def classify_few_shot(email_text: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": FEW_SHOT_SYSTEM},
            {"role": "user",   "content": f"Classify this email:\n\n{email_text}"}
        ],
        temperature=0,
        max_tokens=10,
    )
    pred = response.choices[0].message.content.strip().lower()
    for label in ["ai_phishing", "traditional_phishing", "legitimate"]:
        if label in pred:
            return label
    return "legitimate"


print("Running few-shot classification...\n")
few_shot_results = []

for ex in TEST_EMAILS:
    pred = classify_few_shot(ex["text"])
    few_shot_results.append(pred)
    time.sleep(0.5)

few_shot_acc = sum(p == e["label"] for p, e in zip(few_shot_results, TEST_EMAILS)) / len(TEST_EMAILS)

for i, (ex, pred) in enumerate(zip(TEST_EMAILS, few_shot_results)):
    correct = "✅" if pred == ex["label"] else "❌"
    status  = "CORRECT" if pred == ex["label"] else "WRONG"
    print(f"{correct} #{i+1} [{status}]")
    print(f"   Email:     {ex['text']}")
    print(f"   True:      {ex['label']}")
    print(f"   Predicted: {pred}")
    print()

print(f"Few-shot accuracy:  {few_shot_acc*100:.0f}%")
print(f"Zero-shot accuracy: {zero_shot_acc*100:.0f}%")
print(f"\nImprovement from just adding examples to the prompt: {(few_shot_acc - zero_shot_acc)*100:.0f} percentage points")

## What just happened

Same model. No retraining. No new add-ons. No parameter updates.

We just added 6 examples to the prompt, 2 per class and the model's behavior changed. That's it.

The examples work as an anchor. Instead of the model guessing what "ai_phishing" means in the abstract, it now has something concrete to compare against. It sees the pattern, it calibrates, and it generalizes.

For a security team this could be huge. A new phishing campaign hits on Monday morning. You collect 2-3 examples from the emails that got through. You add them to the prompt. By Monday afternoon your detection is already better, no retraining cycle, no waiting on engineering, no new deployment.

That's the real difference between zero-shot and few-shot in production. Now it's your turn.

---

# Build Your Own Detection Prompt

This is where it gets hands-on. You've seen zero-shot fail on AI phishing and few-shot improve things. Now I'll try to help you to build your own detection system from scratch designing the prompt, testing it, finding where it breaks, and iterating.

This is what a Security/AI engineer actually does when deploying a prompt-based classifier in production. You don't just copy a template. You write it, test it against real examples, understand the failures, and improve it.

## The real challenge

Your goal: write a system prompt that scores higher than the zero-shot baseline on the 9 Owlsight test emails.

You have full control over:
- The task description
- The label names and definitions
- The context you give the model about Owlsight
- How many examples you include and which ones
- How you structure the instructions

The only constraint: you can only use Groq. No training, no fine-tuning. Just the prompt.

## Step 1 | Write your prompt system from scratch

Before looking at the baseline prompt, write your own. Don't overthink it, just describe the task the way you'd explain it to a new analyst.

A good system prompt for classification usually has:
- **Role:** who the model is
- **Task:** what it needs to do
- **Classes:** what each label means, with enough context to distinguish them
- **Output format:** exactly what to return

Write yours in the cell below. Don't add examples yet, we'll cover that step later on the module.

In [ ]:
# Think about: who is the model? what does it know about Owlsight or any fictional company, you decide?
# What makes each class different from the others?

MY_SYSTEM_PROMPT = """
TODO: Write your system prompt here.
"""

# Test it against all 9 emails
def classify_custom(email_text: str, system_prompt: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": f"Classify this email:\n\n{email_text}"}
        ],
        temperature=0,
        max_tokens=10,
    )
    pred = response.choices[0].message.content.strip().lower()
    for label in ["ai_phishing", "traditional_phishing", "legitimate"]:
        if label in pred:
            return label
    return "legitimate"


print("Running your custom prompt...\n")
my_results = []

for ex in TEST_EMAILS:
    pred = classify_custom(ex["text"], MY_SYSTEM_PROMPT)
    my_results.append(pred)
    time.sleep(0.5)

my_acc = sum(p == e["label"] for p, e in zip(my_results, TEST_EMAILS)) / len(TEST_EMAILS)

for i, (ex, pred) in enumerate(zip(TEST_EMAILS, my_results)):
    correct = "✅" if pred == ex["label"] else "❌"
    status  = "CORRECT" if pred == ex["label"] else "WRONG"
    print(f"{correct} #{i+1} [{status}]")
    print(f"   Email:     {ex['text']}")
    print(f"   True:      {ex['label']}")
    print(f"   Predicted: {pred}")
    print()

print(f"Your prompt accuracy:    {my_acc*100:.0f}%")
print(f"Zero-shot baseline:      {zero_shot_acc*100:.0f}%")
delta = (my_acc - zero_shot_acc) * 100
print(f"Difference:              {'+' if delta >= 0 else ''}{delta:.0f} percentage points")

## Step 2 | Understand your failures

Look at which emails your prompt got wrong. Before adding examples, try to fix them through better instructions.

Common failure patterns and how to address them in the prompt:

| Failure | What's happening | How to fix in the prompt |
|---------|-----------------|-------------------------|
| AI phishing classified as legitimate | Model doesn't know what Owlsight-specific attacks look like | Add context about the company, the people, the tools |
| AI phishing classified as traditional | Model sees urgency and defaults to phishing category | Clarify the distinction: ai_phishing is sophisticated and hard to spot |
| Legitimate classified as phishing | Model is too aggressive | Add examples of legitimate emails that might look suspicious |
| Inconsistent labels | Prompt output format is ambiguous | Be more explicit: "respond with exactly one word" |

Try improving your prompt based on what you get, then re-run the cell above.

## Step 3 | Add few-shot examples strategically

Now add examples. But don't just add random ones, try to be as realistic as possible. The examples you choose should directly address the failures you saw in Step 2.

**The strategy:** pick examples that are as close as possible to the emails your prompt got wrong, but correctly labeled. If the model kept calling X person's email legitimate, add an example of a X person  AI phishing email.

Run the cell below to browse your generated dataset and pick your examples.

In [ ]:
# Browse the generated dataset to find good few-shot examples
# Look for examples that are similar to the ones your prompt got wrong or even ask a model to generate similar ones

print("Sample examples from your dataset by class:\n")

for label in ["legitimate", "traditional_phishing", "ai_phishing"]:
    examples = [ex for ex in dataset if ex["label"] == label]
    print(f"--- {label.upper()} (showing 5 of {len(examples)}) ---")
    for ex in examples[:5]:
        print(f"  {ex['text'][:150]}")
    print()

In [ ]:
# Build your few-shot prompt
# Pick 2-3 examples per class that best represent what each class looks like
# Focus on the hard cases, especially ai_phishing examples that look legitimate

MY_FEW_SHOT_EXAMPLES = [
    # In order to paste the email texts you want to use as examples use the following format:
    # Format: {"text": "...", "label": "legitimate"}

    # Legitimate examples
    {"text": "TODO: paste a legitimate email here", "label": "legitimate"},
    {"text": "TODO: paste another legitimate email here", "label": "legitimate"},

    # Traditional phishing examples
    {"text": "TODO: paste a traditional phishing email here", "label": "traditional_phishing"},
    {"text": "TODO: paste another traditional phishing email here", "label": "traditional_phishing"},

    # AI phishing examples, these are the most important ones
    {"text": "TODO: paste an AI phishing email here", "label": "ai_phishing"},
    {"text": "TODO: paste another AI phishing email here", "label": "ai_phishing"},
]

my_few_shot_block = "\n".join([
    f'Email: "{ex["text"][:200]}"\nLabel: {ex["label"]}\n'
    for ex in MY_FEW_SHOT_EXAMPLES
])

MY_FEW_SHOT_SYSTEM = f"""{MY_SYSTEM_PROMPT}

Here are examples of each class:

{my_few_shot_block}
Respond with exactly one of the three labels and nothing else."""

print("Your few-shot prompt preview:\n")
print(MY_FEW_SHOT_SYSTEM[:800] + "...")

In [ ]:
print("Running your few-shot prompt...\n")
my_few_shot_results = []

for ex in TEST_EMAILS:
    pred = classify_custom(ex["text"], MY_FEW_SHOT_SYSTEM)
    my_few_shot_results.append(pred)
    time.sleep(0.5)

my_few_shot_acc = sum(p == e["label"] for p, e in zip(my_few_shot_results, TEST_EMAILS)) / len(TEST_EMAILS)

for i, (ex, pred) in enumerate(zip(TEST_EMAILS, my_few_shot_results)):
    correct = "✅" if pred == ex["label"] else "❌"
    status  = "CORRECT" if pred == ex["label"] else "WRONG"
    print(f"{correct} #{i+1} [{status}]")
    print(f"   Email:     {ex['text']}")
    print(f"   True:      {ex['label']}")
    print(f"   Predicted: {pred}")
    print()

## Your own scoreboard

In [ ]:
print("=" * 60)
print("  FINAL SCOREBOARD")
print("=" * 60)
print(f"  {'Zero-shot baseline':<35} {zero_shot_acc*100:.0f}%")
print(f"  {'Reference few-shot (2 examples/class)':<35} {few_shot_acc*100:.0f}%")
print(f"  {'Your zero-shot prompt':<35} {my_acc*100:.0f}%")
print(f"  {'Your few-shot prompt':<35} {my_few_shot_acc*100:.0f}%")
print("=" * 60)
print()

best = max(zero_shot_acc, few_shot_acc, my_acc, my_few_shot_acc)
if my_few_shot_acc == best:
    print("🏆 Your few-shot prompt is the best performing one.")
elif my_acc == best:
    print("🏆 Your zero-shot prompt is the best, examples didn't help further.")
elif few_shot_acc == best:
    print("The reference few-shot is still winning, look at which examples it used vs yours.")
else:
    print("The zero-shot baseline is still on top, prompt engineering is harder than it looks.")

## What this exercise teaches you

Writing a prompt from scratch and watching it fail is more valuable than copying a working one. You now know:

- **Label definitions matter:** poor defined labels produce inconsistent outputs. The more precisely you define each class, the better.
- **Context anchors the model:** a prompt that mentions Owlsight, Daniel, Priya, and the Series A performs differently than one that says a basic "cybersecurity company"
- **Example selection is a skill:** randomly chosen examples help less than strategically chosen ones that target your specific failure cases
- **Output format is non-negotiable:** if you don't specify exactly what to return, you'll get unpredictable responses that break your parser, or follow my BACP Rule = BE AS CLEAR AS POSSIBLE

These aren't abstract lessons. Every time you deploy a prompt-based classifier in production, you go through exactly this cycle: write, test, find failures, iterate.

---

# Key Takeaways

## What we built

1. **A real threat scenario:** Owlsight, a security company targeted by the exact kind of attack they protect others from
2. **A mixed dataset:** real phishing emails from Kaggle combined with AI-generated Owlsight-specific attacks that no public dataset covers yet
3. **A working detector:** zero-shot and few-shot classifiers running against real test emails
4. **Your own prompt:** written from scratch, tested, iterated, and scored against the baseline

## The thing that actually matters here

Few-shot prompting is not a workaround for not having enough data. It's a deliberate strategy for situations where the threat evolves faster than you can retrain.

In cybersecurity that's almost always the situation. Attackers adapt in days or more realistically in hours. Model retraining cycles take weeks. The gap between those two timelines is where few-shot lives, and now you know how to use it.

## Keep going with your test cases TEST TEST TEST

- Add a 4th class: `business_email_compromise`, wire transfer fraud, vendor impersonation. Can you describe it well enough in a prompt that the model catches it with zero examples?
- Try 5 examples per class instead of 2, does accuracy improve? At what point do you hit diminishing returns?
- Swap `llama-3.1-8b-instant` for `llama-3.1-70b-versatile` on Groq, how much does model size matter for this specific task?
- Take the prompt that performed best and stress test it, write 10 new AI phishing emails yourself and see how many it catches

---

## Next module

**[Module 2 — Chain-of-Thought Prompting →](../../02_chain_of_thought/notebooks/02_chain_of_thought.ipynb)**